# ⚽ Pass Network & Team Structure Analysis
**Julie Landrevie** — Football Data & Video Analyst

**Source :** StatsBomb Open Data | **Stack :** Python · mplsoccer · NetworkX · pandas

> Analyse des réseaux de passes, identification des joueurs-pivots et comparaison de la structure tactique de deux équipes.

---

**Ce notebook couvre :**
1. 📦 Import & chargement des données StatsBomb
2. 🕸️ Construction du réseau de passes
3. 📊 Métriques de centralité (NetworkX)
4. 🌡️ Heatmap de positions moyennes
5. ⚔️ Comparaison entre deux équipes
6. 💾 Export des résultats


## 1. 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import networkx as nx
from mplsoccer import Pitch
from statsbombpy import sb
import warnings
warnings.filterwarnings("ignore")

# ── Style matplotlib (dark, cohérent avec le dashboard Streamlit)
plt.rcParams.update({
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor':   '#0a0e1a',
    'axes.edgecolor':   '#2d3748',
    'axes.labelcolor':  '#a0aec0',
    'xtick.color':      '#718096',
    'ytick.color':      '#718096',
    'text.color':       '#e8eaf0',
    'grid.color':       '#2d3748',
    'font.family':      'DejaVu Sans',
})

# Palette identique au dashboard
DARK_BG   = '#0a0e1a'
PITCH_CLR = '#2d3748'
ACCENT    = '#4fd1c5'   # teal — équipe 1
ACCENT2   = '#f6ad55'   # orange — équipe 2
LABEL_CLR = '#FFE066'   # jaune — noms joueurs sur le terrain

print("✅ Imports OK")


## 2. 📥 Chargement des données StatsBomb

StatsBomb met à disposition des données open data gratuites pour plusieurs compétitions.
Le SDK `statsbombpy` permet de les charger en une ligne.


In [ ]:
# Toutes les compétitions disponibles
comps = sb.competitions()
comps[['competition_id', 'competition_name', 'season_id', 'season_name']].head(20)


In [ ]:
# ── Sélection : La Liga 2020/2021
# Modifie ces valeurs pour analyser une autre compétition
COMPETITION_ID = 11   # La Liga
SEASON_ID      = 90   # 2020/2021

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
matches[['match_id', 'home_team', 'away_team', 'home_score', 'away_score']].head(10)


In [ ]:
# ── Chargement d'un match
# Change match_index pour analyser un autre match (0 = premier de la liste)
match_index = 0
MATCH_ID    = int(matches.iloc[match_index]['match_id'])

row = matches.iloc[match_index]
print(f"Match sélectionné : {row['home_team']} {row['home_score']} – {row['away_score']} {row['away_team']}")
print(f"Match ID : {MATCH_ID}")

events  = sb.events(match_id=MATCH_ID)
lineups = sb.lineups(match_id=MATCH_ID)
teams   = events['team'].unique().tolist()
TEAM1, TEAM2 = teams[0], teams[1]

print(f"\nÉquipes : {TEAM1} vs {TEAM2}")
print(f"Événements chargés : {len(events):,}")


## 3. 🕸️ Construction du réseau de passes

Un réseau de passes se compose de :
- **Nœuds** : chaque joueur, positionné à sa position moyenne de passe
- **Arêtes** : les liaisons entre deux joueurs, dont l'épaisseur représente le nombre de passes échangées

On ne garde que les **passes complétées** pour refléter le jeu effectif.


In [ ]:
def build_pass_network(events, team, period='Full Match', min_passes=3):
    """
    Construit nodes (positions moyennes) et edges (liaisons de passes).

    Parameters
    ----------
    events     : pd.DataFrame  — événements StatsBomb du match
    team       : str           — nom de l'équipe
    period     : str           — 'Full Match' | '1st Half' | '2nd Half'
    min_passes : int           — seuil minimum de passes par liaison

    Returns
    -------
    nodes : pd.DataFrame  — avg_x, avg_y, pass_count par joueur
    edges : pd.DataFrame  — nombre de passes par paire (player → recipient)
    """
    # Passes complétées uniquement (pass_outcome == NaN = succès dans StatsBomb)
    passes = events[
        (events['type'] == 'Pass') &
        (events['team'] == team) &
        (events['pass_outcome'].isna())
    ].copy()

    # Filtre période
    if period == '1st Half':
        passes = passes[passes['period'] == 1]
    elif period == '2nd Half':
        passes = passes[passes['period'] == 2]

    # Extraction des coordonnées
    passes['x']     = passes['location'].apply(lambda l: l[0] if isinstance(l, list) else np.nan)
    passes['y']     = passes['location'].apply(lambda l: l[1] if isinstance(l, list) else np.nan)
    passes['end_x'] = passes['pass_end_location'].apply(lambda l: l[0] if isinstance(l, list) else np.nan)
    passes['end_y'] = passes['pass_end_location'].apply(lambda l: l[1] if isinstance(l, list) else np.nan)
    passes = passes.dropna(subset=['x', 'y', 'end_x', 'end_y', 'pass_recipient'])

    # XI de départ : résolu depuis les events de période 1 (avant le filtre période)
    # ⚠️ Important : on cherche les titulaires sur TOUS les events, pas seulement ceux filtrés
    all_p1 = events[
        (events['type'] == 'Pass') &
        (events['team'] == team) &
        (events['pass_outcome'].isna()) &
        (events['period'] == 1)
    ]
    starters = all_p1['player'].value_counts()
    xi = starters[starters >= 1].index.tolist()[:11]

    # Positions moyennes (nœuds)
    nodes = (
        passes[passes['player'].isin(xi)]
        .groupby('player')
        .agg(avg_x=('x', 'mean'), avg_y=('y', 'mean'), pass_count=('x', 'count'))
        .reset_index()
    )

    # Liaisons (arêtes)
    pairs = passes[passes['player'].isin(xi) & passes['pass_recipient'].isin(xi)]
    edges = pairs.groupby(['player', 'pass_recipient']).size().reset_index(name='count')
    edges = edges[edges['count'] >= min_passes]

    return nodes, edges


# Construction des réseaux pour les deux équipes
nodes1, edges1 = build_pass_network(events, TEAM1)
nodes2, edges2 = build_pass_network(events, TEAM2)

print(f"{TEAM1} : {len(nodes1)} nœuds · {len(edges1)} liaisons")
print(f"{TEAM2} : {len(nodes2)} nœuds · {len(edges2)} liaisons")


## 4. 📊 Métriques de centralité (NetworkX)

### Betweenness Centrality
Mesure combien de fois un joueur se trouve sur le **chemin le plus court** entre deux autres joueurs.
Un score élevé = joueur-pivot dont la perte fragilise l'organisation collective.

### Density
Ratio arêtes existantes / arêtes possibles. Une densité de 1 = tout le monde passe à tout le monde.

### Clustering
Tendance à jouer en **triangles** (possession en triangle = possession sécurisée).


In [ ]:
def compute_centrality(nodes, edges):
    """Ajoute betweenness centrality et in/out degree aux nœuds."""
    if nodes.empty or edges.empty:
        return nodes

    G = nx.DiGraph()
    for _, r in nodes.iterrows():
        G.add_node(r['player'])
    for _, r in edges.iterrows():
        G.add_edge(r['player'], r['pass_recipient'], weight=r['count'])

    bc  = nx.betweenness_centrality(G, weight='weight', normalized=True)
    ind = dict(G.in_degree(weight='weight'))
    oud = dict(G.out_degree(weight='weight'))

    nodes = nodes.copy()
    nodes['betweenness'] = nodes['player'].map(bc).fillna(0)
    nodes['in_degree']   = nodes['player'].map(ind).fillna(0)
    nodes['out_degree']  = nodes['player'].map(oud).fillna(0)
    return nodes


def team_metrics(nodes, edges):
    """Métriques globales de structure d'équipe."""
    if nodes.empty or edges.empty:
        return {}

    G = nx.DiGraph()
    for _, r in nodes.iterrows(): G.add_node(r['player'])
    for _, r in edges.iterrows(): G.add_edge(r['player'], r['pass_recipient'], weight=r['count'])

    n   = len(G.nodes)
    bc  = list(nx.betweenness_centrality(G, normalized=True).values())
    mbc = max(bc) if bc else 0

    return {
        'density':         round(nx.density(G), 3),
        'clustering':      round(nx.average_clustering(G.to_undirected()), 3),
        'centralization':  round(sum(mbc - v for v in bc) / (n - 1) * 100, 1) if n > 1 else 0,
        'avg_link_passes': round(edges['count'].mean(), 1),
        'total_passes':    int(nodes['pass_count'].sum()),
        'connections':     len(edges),
    }


nodes1 = compute_centrality(nodes1, edges1)
nodes2 = compute_centrality(nodes2, edges2)
m1 = team_metrics(nodes1, edges1)
m2 = team_metrics(nodes2, edges2)

print(f"\n{'Métrique':<22} {TEAM1:<25} {TEAM2}")
print("─" * 70)
for key in ['total_passes', 'connections', 'density', 'clustering', 'centralization', 'avg_link_passes']:
    print(f"{key:<22} {str(m1.get(key, '—')):<25} {m2.get(key, '—')}")


In [ ]:
# Top 5 joueurs par betweenness centrality
for team, nodes in [(TEAM1, nodes1), (TEAM2, nodes2)]:
    print(f"\n🏅 {team} — Pivots tactiques (Betweenness Centrality)")
    print("─" * 55)
    top = nodes.sort_values('betweenness', ascending=False).head(5)
    for i, (_, r) in enumerate(top.iterrows(), 1):
        bar = '█' * int(r['betweenness'] * 40)
        print(f"  {i}. {r['player']:<35} {r['betweenness']:.3f}  {bar}")


## 5. 🎨 Visualisations

### Lecture du réseau de passes
- **Taille du nœud** : nombre de passes effectuées
- **Épaisseur de l'arête** : fréquence de la liaison entre deux joueurs
- **Couleur du label** : jaune (#FFE066) pour la visibilité sur fond sombre


In [ ]:
def clean_name(full_name):
    """Retourne un nom lisible (prénom + nom de famille principal)."""
    NAME_MAP = {
        "Gerard Piqué Bernabéu": "Gerard Piqué",
        "Sergio Busquets i Burgos": "Sergio Busquets",
        "Sergi Roberto Carnicer": "Sergi Roberto",
        "Lionel Andrés Messi Cuccittini": "Lionel Messi",
        "Jordi Alba Ramos": "Jordi Alba",
        "Antoine Griezmann": "Antoine Griezmann",
        "Anssumane Fati Vieira": "Ansu Fati",
        "Rodrigo Andrés Battaglia": "Rodrigo Battaglia",
        "Fernando Pacheco Flores": "Fernando Pacheco",
        "Joaquín Navarro Jiménez": "Joaquín Navarro",
        "Rubén Duarte Sánchez": "Rubén Duarte",
        "José Ignacio Peleteiro Ramallo": "Joselu",
        "Edgar Antonio Méndez Ortega": "Edgar Méndez",
    }
    if full_name in NAME_MAP:
        return NAME_MAP[full_name]
    parts = full_name.strip().split()
    return f"{parts[0]} {parts[-1]}" if len(parts) >= 2 else full_name


def plot_network(nodes, edges, team, accent=ACCENT, ax=None, show=True):
    """Visualise le réseau de passes pour une équipe."""
    pitch = Pitch(pitch_type='statsbomb', pitch_color=DARK_BG,
                  line_color=PITCH_CLR, linewidth=1.5, corner_arcs=True)

    if ax is None:
        fig, ax = pitch.draw(figsize=(13, 8))
        fig.patch.set_facecolor(DARK_BG)
    else:
        pitch.draw(ax=ax)

    if nodes.empty or edges.empty:
        ax.text(60, 34, 'Données insuffisantes', color='white', fontsize=12, ha='center')
        return

    max_p = nodes['pass_count'].max()
    max_e = edges['count'].max()

    # Arêtes
    for _, edge in edges.iterrows():
        fn = nodes[nodes['player'] == edge['player']]
        tn = nodes[nodes['player'] == edge['pass_recipient']]
        if fn.empty or tn.empty: continue
        xs, ys = fn.iloc[0][['avg_x', 'avg_y']]
        xe, ye = tn.iloc[0][['avg_x', 'avg_y']]
        alpha = 0.15 + 0.65 * (edge['count'] / max_e)
        lw    = 0.5  + 5.0  * (edge['count'] / max_e)
        ax.annotate('', xy=(xe, ye), xytext=(xs, ys),
                    arrowprops=dict(arrowstyle='->', color=to_rgba(accent, alpha),
                                   lw=lw, connectionstyle='arc3,rad=0.05'))

    # Nœuds
    for _, node in nodes.iterrows():
        size  = 900 + 2500 * (node['pass_count'] / max_p)
        ax.scatter(node['avg_x'], node['avg_y'], s=size, c=accent,
                   zorder=5, alpha=0.9, linewidths=2, edgecolors='white')
        short = clean_name(node['player'])
        # Label jaune avec fond sombre pour lisibilité
        ax.text(node['avg_x'], node['avg_y'] - 3.8, short,
                color=LABEL_CLR, fontsize=8, fontweight='bold',
                ha='center', va='top', zorder=6,
                bbox=dict(boxstyle='round,pad=0.15', facecolor=DARK_BG, alpha=0.55, edgecolor='none'))
        ax.text(node['avg_x'], node['avg_y'], str(int(node['pass_count'])),
                color=DARK_BG, fontsize=7.5, fontweight='bold',
                ha='center', va='center', zorder=7)

    ax.set_title(team, color='white', fontsize=13, fontweight='bold', pad=10)
    ax.text(1, 80, '● taille nœud = passes effectuées', color='#718096', fontsize=7.5)
    ax.text(1, 77, '→ épaisseur = fréquence de liaison', color='#718096', fontsize=7.5)

    if show and ax is None:
        plt.tight_layout()
        plt.show()


# ── Affichage côte à côte
fig, axes = plt.subplots(1, 2, figsize=(22, 9), facecolor=DARK_BG)
plot_network(nodes1, edges1, TEAM1, accent=ACCENT,  ax=axes[0], show=False)
plot_network(nodes2, edges2, TEAM2, accent=ACCENT2, ax=axes[1], show=False)
fig.suptitle('Pass Networks — Full Match', color='white', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('pass_networks.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("✅ Sauvegardé : pass_networks.png")


In [ ]:
# ── Heatmaps de positions de passe
fig, axes = plt.subplots(1, 2, figsize=(22, 9), facecolor=DARK_BG)

for ax_idx, (team, accent) in enumerate([(TEAM1, ACCENT), (TEAM2, ACCENT2)]):
    pitch = Pitch(pitch_type='statsbomb', pitch_color=DARK_BG,
                  line_color=PITCH_CLR, linewidth=1.5)
    pitch.draw(ax=axes[ax_idx])

    team_passes = events[(events['type'] == 'Pass') & (events['team'] == team)].copy()
    team_passes['x'] = team_passes['location'].apply(lambda l: l[0] if isinstance(l, list) else np.nan)
    team_passes['y'] = team_passes['location'].apply(lambda l: l[1] if isinstance(l, list) else np.nan)
    team_passes = team_passes.dropna(subset=['x', 'y'])

    if not team_passes.empty:
        pitch.kdeplot(team_passes['x'], team_passes['y'], ax=axes[ax_idx],
                     cmap='YlOrRd', fill=True, levels=100, alpha=0.75, bw_adjust=0.7)

    axes[ax_idx].set_title(f'{team} — Passing Heatmap', color='white',
                            fontsize=13, fontweight='bold', pad=10)
    axes[ax_idx].text(52, -4,
        'Zone chaude = là où l'équipe initie ses passes',
        color='#718096', fontsize=8, ha='center')

fig.suptitle('Passing Position Heatmaps — Full Match', color='white', fontsize=15,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('heatmaps.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("✅ Sauvegardé : heatmaps.png")


In [ ]:
# ── Betweenness Centrality — barres horizontales
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=DARK_BG)

for ax_idx, (nodes_t, team, accent) in enumerate([(nodes1, TEAM1, ACCENT), (nodes2, TEAM2, ACCENT2)]):
    ax = axes[ax_idx]
    ax.set_facecolor(DARK_BG)

    df = nodes_t.sort_values('betweenness', ascending=True).tail(11).copy()
    df['label'] = df['player'].apply(clean_name)
    colors = [accent if v == df['betweenness'].max() else '#2d4a5a' for v in df['betweenness']]

    bars = ax.barh(df['label'], df['betweenness'], color=colors, height=0.6)
    for bar, val in zip(bars, df['betweenness']):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', color='#a0aec0', fontsize=9)

    ax.set_xlabel('Betweenness Centrality', color='#718096', fontsize=10)
    ax.set_title(f'{team} — Pivots tactiques', color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='#a0aec0', labelsize=9)
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
    for spine in ['bottom', 'left']: ax.spines[spine].set_color('#2d3748')

plt.tight_layout()
plt.savefig('centrality.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("✅ Sauvegardé : centrality.png")


## 6. ⚔️ Comparaison des structures d'équipe

### Interprétation des métriques

| Métrique | Description | Valeur élevée |
|---|---|---|
| **Density** | Ratio liaisons existantes / possibles | Jeu ultra-connecté (possession) |
| **Clustering** | Fréquence des triangles de passes | Possession en triangle, sécurisée |
| **Centralization** | Concentration du jeu sur un joueur | Dépendance individuelle forte |
| **Avg Passes/Link** | Intensité des liaisons | Connexions fréquemment utilisées |


In [ ]:
# Tableau comparatif complet
import pandas as pd

comp = pd.DataFrame({
    'Metric':    ['Total Passes', 'Active Connections', 'Network Density',
                  'Avg Clustering', 'Centralization (%)', 'Avg Passes/Link'],
    TEAM1:       [m1['total_passes'], m1['connections'], m1['density'],
                  m1['clustering'], f"{m1['centralization']}%", m1['avg_link_passes']],
    TEAM2:       [m2['total_passes'], m2['connections'], m2['density'],
                  m2['clustering'], f"{m2['centralization']}%", m2['avg_link_passes']],
})

print(comp.to_string(index=False))
print()

# Interprétation automatique
print("── Analyse tactique ─────────────────────────────────────────")
if abs(m1['density'] - m2['density']) > 0.05:
    leader = TEAM1 if m1['density'] > m2['density'] else TEAM2
    print(f"Densité : {leader} présente un réseau plus connecté — davantage d'options de passe.")
if abs(m1['clustering'] - m2['clustering']) > 0.05:
    leader = TEAM1 if m1['clustering'] > m2['clustering'] else TEAM2
    print(f"Clustering : {leader} joue davantage en triangles — possession structurée.")
if abs(m1['centralization'] - m2['centralization']) > 5:
    leader = TEAM1 if m1['centralization'] > m2['centralization'] else TEAM2
    other  = TEAM2 if m1['centralization'] > m2['centralization'] else TEAM1
    print(f"Centralisation : {leader} dépend davantage d'un relayeur clé vs {other} (jeu distribué).")


## 7. 💾 Export des résultats

In [ ]:
import os
os.makedirs('data/exports', exist_ok=True)

# Nodes
n1, n2 = nodes1.copy(), nodes2.copy()
n1['team'], n2['team'] = TEAM1, TEAM2
all_nodes = pd.concat([n1, n2], ignore_index=True)
all_nodes['player_clean'] = all_nodes['player'].apply(clean_name)
all_nodes.to_csv('data/exports/pass_network_nodes.csv', index=False)

# Edges
e1, e2 = edges1.copy(), edges2.copy()
e1['team'], e2['team'] = TEAM1, TEAM2
all_edges = pd.concat([e1, e2], ignore_index=True)
all_edges.to_csv('data/exports/pass_network_edges.csv', index=False)

# Team metrics
pd.DataFrame([{'team': TEAM1, **m1}, {'team': TEAM2, **m2}]).to_csv(
    'data/exports/team_metrics.csv', index=False)

print("✅ Export dans data/exports/ :")
print("   pass_network_nodes.csv")
print("   pass_network_edges.csv")
print("   team_metrics.csv")
